# Random Forest

This notebook owns the initial model, tuning, training/validation diagnostics, and saved fitted pipelines for Random Forest.

Two scenarios use the same 395 students (including 38 zero grades), 316/79 split, and five training-validation folds:

- **Without prior grades:** G1 and G2 are excluded. This remains the main project scenario; availability of other features, especially absences, still requires review.
- **With prior grades:** both G1 and G2 are included. This is a later-stage prediction scenario, available only after the second-period grade is known. It is not an improvement to an earlier prediction at the same time.

MAE is the primary selection metric; RMSE and R2 are supporting metrics. No held-out test performance is calculated here. The full dataset was explored in Notebook 01, so the holdout is not fully blind to exploratory analysis.

Random Forest averages 300 randomized trees. The search varies depth, leaf size, and feature sampling while holding tree count fixed. Parallelism is handled by cross-validation/search; each forest uses one worker to avoid nested parallelism.

## Setup and Reproducible Split

In [1]:
MODEL_NAME = 'Random Forest'
MODEL_SLUG = 'random_forest'
SCALE_NUMERIC = False

from sklearn.ensemble import RandomForestRegressor
INITIAL_ESTIMATOR = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.8, random_state=42, n_jobs=1)
MODEL_PARAM_GRID = {'model__max_depth': [None, 5, 10], 'model__min_samples_leaf': [2, 5, 10], 'model__max_features': [0.8, 1.0]}

In [2]:
from pathlib import Path
import sys
import hashlib
import json
import platform

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn import set_config
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

set_config(display="text")

# Locate the repository from either its root or a nested notebook folder.
PROJECT_ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data" / "student-mat.csv").is_file()
    and (folder / "src" / "preprocessing.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from preprocessing import make_preprocessor

DATA_PATH = PROJECT_ROOT / "data" / "student-mat.csv"
CODE_PATH = PROJECT_ROOT / "src" / "preprocessing.py"
df = pd.read_csv(DATA_PATH, sep=";")
assert df["G3"].notna().all()
assert df["G3"].between(0, 20).all()
assert len(df) == 395 and df["G3"].eq(0).sum() == 38

train_indices, test_indices = train_test_split(
    df.index.to_numpy(), test_size=0.2, random_state=42
)
assert not set(train_indices) & set(test_indices)
y_train = df.loc[train_indices, "G3"]

# Only the row indices of the reserved test set are needed here.
# Do not evaluate predictions against test grades during model selection.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}
metrics = ["MAE", "RMSE", "R2"]

def fold_results(scores, partition="test"):
    return pd.DataFrame({
        "Fold": np.arange(1, cv.get_n_splits() + 1),
        "MAE": -scores[f"{partition}_MAE"],
        "RMSE": -scores[f"{partition}_RMSE"],
        "R2": scores[f"{partition}_R2"]
    })

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__
}
experiment = {
    "model": MODEL_NAME,
    "slug": MODEL_SLUG,
    "dataset_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "preprocessing_sha256": hashlib.sha256(CODE_PATH.read_bytes()).hexdigest(),
    "versions": versions,
    "students": len(df),
    "zero_grades": int(df["G3"].eq(0).sum()),
    "training_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
    "fold_validation_indices": [train_indices[v].tolist() for _, v in cv.split(train_indices)],
    "split_random_state": 42,
    "test_size": 0.2,
    "selection_metric": "MAE",
    "test_performance_evaluated": False,
    "scenarios": {}
}
print("Students:", len(df), "| Zero grades retained:", experiment["zero_grades"])
print("Training:", len(train_indices), "| Reserved test:", len(test_indices))

Students: 395 | Zero grades retained: 38
Training: 316 | Reserved test: 79


## Constant Baselines

The mean is a conventional benchmark. The median is also included because it minimizes absolute error for a constant prediction. Each baseline learns from the training portion of each fold.

In [3]:
# Both baselines ignore inputs, so the same results apply to both scenarios.
X_baseline = df.loc[train_indices].drop(columns=["G3", "G1", "G2"])
baseline_results = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(
        DummyRegressor(strategy=strategy),
        X_baseline, y_train, cv=cv, scoring=scoring, error_score="raise"
    )
    baseline_results[strategy] = fold_results(scores)

baseline_comparison = pd.DataFrame({
    name: folds[metrics].mean() for name, folds in baseline_results.items()
}).T
baseline_comparison

,MAE,RMSE,R2
mean,3.402865,4.563458,-0.035023
median,3.340278,4.593095,-0.051899


## Without G1 and G2

All learned preprocessing is fitted inside each cross-validation training fold.

In [4]:
scenario_id = 'no_prior_grades'
include_prior_grades = False
feature_columns = [
    column for column in df.columns
    if column != "G3" and (include_prior_grades or column not in ["G1", "G2"])
]
X_train = df.loc[train_indices, feature_columns].copy()
assert X_train.index.equals(y_train.index)
assert "G3" not in X_train.columns
assert ({"G1", "G2"} <= set(X_train.columns)) == include_prior_grades
print("Scenario:", scenario_id, "| Original training inputs:", X_train.shape)

Scenario: no_prior_grades | Original training inputs: (316, 30)


### Initial Model

Start with original absences. These settings are provisional.

In [5]:
base_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=SCALE_NUMERIC,
    include_prior_grades=include_prior_grades
)
base_preprocessor.set_params(column_processing__nominal__drop="first")
base_pipeline = Pipeline([
    ("preprocessing", base_preprocessor),
    ("model", clone(INITIAL_ESTIMATOR))
])

base_scores = cross_validate(
    base_pipeline, X_train, y_train,
    cv=cv, scoring=scoring, return_train_score=True,
    n_jobs=-1, error_score="raise"
)
base_folds = fold_results(base_scores)
display(base_folds)
display(base_folds[metrics].agg(["mean", "std"]))

,Fold,MAE,RMSE,R2
0,1,2.595719,3.554345,0.120141
1,2,3.553678,4.434744,0.346881
2,3,2.749193,3.676339,0.037727
3,4,3.091701,4.013865,0.190856
4,5,3.017331,4.083242,0.329864


,MAE,RMSE,R2
mean,3.001524,3.952507,0.205094
std,0.367969,0.349360,0.133325


### Search and Selection

Compare both absence representations and the model settings defined above. The search includes the initial configuration. The best combination is selected by MAE and refitted on all 316 training students.

In [6]:
preprocessing_options = []
for use_log in [False, True]:
    preprocessor = make_preprocessor(
        use_log_absences=use_log,
        scale_numeric=SCALE_NUMERIC,
        include_prior_grades=include_prior_grades
    )
    preprocessor.set_params(column_processing__nominal__drop="first")
    preprocessing_options.append(preprocessor)

parameter_grid = dict(MODEL_PARAM_GRID)
parameter_grid["preprocessing"] = preprocessing_options
search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    error_score="raise"
)
search.fit(X_train, y_train)

best_index = search.best_index_
best_pipeline = search.best_estimator_
tuned_scores = {
    f"{partition}_{metric}": np.array([
        search.cv_results_[f"split{i}_{partition}_{metric}"][best_index]
        for i in range(cv.get_n_splits())
    ])
    for partition in ["train", "test"]
    for metric in metrics
}
tuned_folds = fold_results(tuned_scores)
best_log = best_pipeline.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"]
best_parameters = {key: value for key, value in search.best_params_.items() if key.startswith("model__")}
print("Selected model parameters:", best_parameters or "No model parameters searched (OLS).")
print("Selected log absences:", best_log)
print("Candidate combinations:", len(search.cv_results_["params"]))
display(tuned_folds[metrics].agg(["mean", "std"]))

Selected model parameters: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 2}
Selected log absences: False
Candidate combinations: 36


,MAE,RMSE,R2
mean,2.964054,3.936634,0.208057
std,0.345624,0.320410,0.153841


### Comparison and Training Gap

Training/validation gaps can reveal overfitting. These validation scores have been used for selection and may be optimistic; they are not an independent test evaluation.

In [7]:
model_comparison = pd.DataFrame({
    "Mean baseline": baseline_results["mean"][metrics].mean(),
    "Median baseline": baseline_results["median"][metrics].mean(),
    "Initial model": base_folds[metrics].mean(),
    "Selected candidate": tuned_folds[metrics].mean()
}).T

generalization_diagnostic = pd.DataFrame({
    "Initial training": fold_results(base_scores, "train")[metrics].mean(),
    "Initial validation": base_folds[metrics].mean(),
    "Selected training": fold_results(tuned_scores, "train")[metrics].mean(),
    "Selected validation": tuned_folds[metrics].mean()
}).T

display(model_comparison)
display(generalization_diagnostic)

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Median baseline,3.340278,4.593095,-0.051899
Initial model,3.001524,3.952507,0.205094
Selected candidate,2.964054,3.936634,0.208057


,MAE,RMSE,R2
Initial training,2.107394,2.811236,0.622175
Initial validation,3.001524,3.952507,0.205094
Selected training,1.375272,1.886727,0.829703
Selected validation,2.964054,3.936634,0.208057


### Save This Scenario

Save the complete fitted preprocessing-and-model pipeline. The metadata records the exact inputs and checks that prior grades are actually included or excluded as intended.

In [8]:
column_transformer = best_pipeline.named_steps["preprocessing"].named_steps["column_processing"]
encoded_feature_names = column_transformer.get_feature_names_out().tolist()
for grade in ["G1", "G2"]:
    assert (f"numeric_ordinal__{grade}" in encoded_feature_names) == include_prior_grades
assert not any(name.endswith("__G3") for name in encoded_feature_names)

artifact_path = PROJECT_ROOT / "models" / "experiments" / f"{MODEL_SLUG}__{scenario_id}.joblib"
artifact_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, artifact_path)

experiment["scenarios"][scenario_id] = {
    "include_prior_grades": include_prior_grades,
    "input_features": feature_columns,
    "encoded_features": encoded_feature_names,
    "initial_scores": base_folds[metrics].mean().to_dict(),
    "initial_std": base_folds[metrics].std().to_dict(),
    "selected_scores": tuned_folds[metrics].mean().to_dict(),
    "selected_std": tuned_folds[metrics].std().to_dict(),
    "initial_folds": base_folds.to_dict(orient="records"),
    "selected_folds": tuned_folds.to_dict(orient="records"),
    "mean_baseline_scores": baseline_results["mean"][metrics].mean().to_dict(),
    "median_baseline_scores": baseline_results["median"][metrics].mean().to_dict(),
    "training_validation_diagnostic": generalization_diagnostic.to_dict(orient="index"),
    "selected_model_parameters": best_parameters,
    "log_absences": best_log,
    "scaling": SCALE_NUMERIC,
    "search_candidates": len(search.cv_results_["params"]),
    "pipeline_path": artifact_path.relative_to(PROJECT_ROOT).as_posix(),
    "pipeline_sha256": hashlib.sha256(artifact_path.read_bytes()).hexdigest()
}
print("Saved fitted training pipeline:", artifact_path.relative_to(PROJECT_ROOT))

Saved fitted training pipeline: models\experiments\random_forest__no_prior_grades.joblib


## With G1 and G2

All learned preprocessing is fitted inside each cross-validation training fold.

In [9]:
scenario_id = 'with_prior_grades'
include_prior_grades = True
feature_columns = [
    column for column in df.columns
    if column != "G3" and (include_prior_grades or column not in ["G1", "G2"])
]
X_train = df.loc[train_indices, feature_columns].copy()
assert X_train.index.equals(y_train.index)
assert "G3" not in X_train.columns
assert ({"G1", "G2"} <= set(X_train.columns)) == include_prior_grades
print("Scenario:", scenario_id, "| Original training inputs:", X_train.shape)

Scenario: with_prior_grades | Original training inputs: (316, 32)


### Initial Model

Start with original absences. These settings are provisional.

In [10]:
base_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=SCALE_NUMERIC,
    include_prior_grades=include_prior_grades
)
base_preprocessor.set_params(column_processing__nominal__drop="first")
base_pipeline = Pipeline([
    ("preprocessing", base_preprocessor),
    ("model", clone(INITIAL_ESTIMATOR))
])

base_scores = cross_validate(
    base_pipeline, X_train, y_train,
    cv=cv, scoring=scoring, return_train_score=True,
    n_jobs=-1, error_score="raise"
)
base_folds = fold_results(base_scores)
display(base_folds)
display(base_folds[metrics].agg(["mean", "std"]))

,Fold,MAE,RMSE,R2
0,1,0.899495,1.315700,0.879439
1,2,1.362783,2.260032,0.830377
2,3,0.829711,1.228340,0.892575
3,4,0.828176,1.195869,0.928176
4,5,1.002853,1.779714,0.872693


,MAE,RMSE,R2
mean,0.984604,1.555931,0.880652
std,0.223079,0.458396,0.035331


### Search and Selection

Compare both absence representations and the model settings defined above. The search includes the initial configuration. The best combination is selected by MAE and refitted on all 316 training students.

In [11]:
preprocessing_options = []
for use_log in [False, True]:
    preprocessor = make_preprocessor(
        use_log_absences=use_log,
        scale_numeric=SCALE_NUMERIC,
        include_prior_grades=include_prior_grades
    )
    preprocessor.set_params(column_processing__nominal__drop="first")
    preprocessing_options.append(preprocessor)

parameter_grid = dict(MODEL_PARAM_GRID)
parameter_grid["preprocessing"] = preprocessing_options
search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    error_score="raise"
)
search.fit(X_train, y_train)

best_index = search.best_index_
best_pipeline = search.best_estimator_
tuned_scores = {
    f"{partition}_{metric}": np.array([
        search.cv_results_[f"split{i}_{partition}_{metric}"][best_index]
        for i in range(cv.get_n_splits())
    ])
    for partition in ["train", "test"]
    for metric in metrics
}
tuned_folds = fold_results(tuned_scores)
best_log = best_pipeline.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"]
best_parameters = {key: value for key, value in search.best_params_.items() if key.startswith("model__")}
print("Selected model parameters:", best_parameters or "No model parameters searched (OLS).")
print("Selected log absences:", best_log)
print("Candidate combinations:", len(search.cv_results_["params"]))
display(tuned_folds[metrics].agg(["mean", "std"]))

Selected model parameters: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 2}
Selected log absences: True
Candidate combinations: 36


,MAE,RMSE,R2
mean,0.938128,1.475705,0.892770
std,0.205599,0.479702,0.036863


### Comparison and Training Gap

Training/validation gaps can reveal overfitting. These validation scores have been used for selection and may be optimistic; they are not an independent test evaluation.

In [12]:
model_comparison = pd.DataFrame({
    "Mean baseline": baseline_results["mean"][metrics].mean(),
    "Median baseline": baseline_results["median"][metrics].mean(),
    "Initial model": base_folds[metrics].mean(),
    "Selected candidate": tuned_folds[metrics].mean()
}).T

generalization_diagnostic = pd.DataFrame({
    "Initial training": fold_results(base_scores, "train")[metrics].mean(),
    "Initial validation": base_folds[metrics].mean(),
    "Selected training": fold_results(tuned_scores, "train")[metrics].mean(),
    "Selected validation": tuned_folds[metrics].mean()
}).T

display(model_comparison)
display(generalization_diagnostic)

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Median baseline,3.340278,4.593095,-0.051899
Initial model,0.984604,1.555931,0.880652
Selected candidate,0.938128,1.475705,0.892770


,MAE,RMSE,R2
Initial training,0.705083,1.137001,0.938196
Initial validation,0.984604,1.555931,0.880652
Selected training,0.448793,0.743149,0.973560
Selected validation,0.938128,1.475705,0.892770


### Save This Scenario

Save the complete fitted preprocessing-and-model pipeline. The metadata records the exact inputs and checks that prior grades are actually included or excluded as intended.

In [13]:
column_transformer = best_pipeline.named_steps["preprocessing"].named_steps["column_processing"]
encoded_feature_names = column_transformer.get_feature_names_out().tolist()
for grade in ["G1", "G2"]:
    assert (f"numeric_ordinal__{grade}" in encoded_feature_names) == include_prior_grades
assert not any(name.endswith("__G3") for name in encoded_feature_names)

artifact_path = PROJECT_ROOT / "models" / "experiments" / f"{MODEL_SLUG}__{scenario_id}.joblib"
artifact_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, artifact_path)

experiment["scenarios"][scenario_id] = {
    "include_prior_grades": include_prior_grades,
    "input_features": feature_columns,
    "encoded_features": encoded_feature_names,
    "initial_scores": base_folds[metrics].mean().to_dict(),
    "initial_std": base_folds[metrics].std().to_dict(),
    "selected_scores": tuned_folds[metrics].mean().to_dict(),
    "selected_std": tuned_folds[metrics].std().to_dict(),
    "initial_folds": base_folds.to_dict(orient="records"),
    "selected_folds": tuned_folds.to_dict(orient="records"),
    "mean_baseline_scores": baseline_results["mean"][metrics].mean().to_dict(),
    "median_baseline_scores": baseline_results["median"][metrics].mean().to_dict(),
    "training_validation_diagnostic": generalization_diagnostic.to_dict(orient="index"),
    "selected_model_parameters": best_parameters,
    "log_absences": best_log,
    "scaling": SCALE_NUMERIC,
    "search_candidates": len(search.cv_results_["params"]),
    "pipeline_path": artifact_path.relative_to(PROJECT_ROOT).as_posix(),
    "pipeline_sha256": hashlib.sha256(artifact_path.read_bytes()).hexdigest()
}
print("Saved fitted training pipeline:", artifact_path.relative_to(PROJECT_ROOT))

Saved fitted training pipeline: models\experiments\random_forest__with_prior_grades.joblib


## Scenario Summary

Compare these as predictions made with different information available. Do not choose an early-support model solely because the later-stage scenario has a better score.

In [14]:
scenario_comparison = pd.DataFrame({
    name: result["selected_scores"] for name, result in experiment["scenarios"].items()
}).T
scenario_comparison

,MAE,RMSE,R2
no_prior_grades,2.964054,3.936634,0.208057
with_prior_grades,0.938128,1.475705,0.892770


## Save Experiment Record

The record contains fold scores, selected settings, versions, fingerprints, and shared row indices. Run the central comparison notebook after all five model notebooks.

In [15]:
assert set(experiment["scenarios"]) == {"no_prior_grades", "with_prior_grades"}
result_path = PROJECT_ROOT / "docs" / "experiments" / "per_model" / f"{MODEL_SLUG}.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
result_path.write_text(json.dumps(experiment, indent=2) + "\n", encoding="utf-8")
print("Saved experiment:", result_path.relative_to(PROJECT_ROOT))
print("Held-out test performance has not been evaluated.")

Saved experiment: docs\experiments\per_model\random_forest.json
Held-out test performance has not been evaluated.
